# 构建一个分词器

上一节课程给你一个玩具。这节课给你能用的武器。

## 问题描述

上一节课写的BPE 分词器作用于英文文本。而生产级的分词器要处理任意编码格式的原始字节，所以需要在切分前归一化Unicode，处理不会被合并的特殊词元，把预分词和子词切分串起来，并且要足够快，才能应对大批量的训练语料。

## 基本概念

流水线：

|流程|干什么|为什么|
|---|---|---|
|归一化|把写法不同，但其实是一个字的东西，先统一|不做的话同一句话会被切成不同的词元，模型会当成两个词|
|预切分|先按规则/正则把文本切成小块|防止BPE在合并的时候跨越词分解，比如`the cat`出现`e c`这种词元|
|BPE合并|在每个小块内部，按训练好的规则，把常一起出现的字节对合并子词|真正压缩的一步|
|特殊token|插入[BOS], [EOS]... 等结构符号|模型靠它们认哪里是开头，哪里是结尾，谁在说话|
|映射成ID|把子词字符串替换成整数|模型只吃数字，不吃字，这才是送进网络的最终输入|



# 开始编码

In [1]:
import re
import unicodedata
from collections import Counter

try:
    import regex
    GPT2_PATTERN = regex.compile(
        r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    )
except ImportError:
    GPT2_PATTERN = re.compile(
        r"""'(?:[sdmt]|ll|ve|re)| ?[a-zA-Z]+| ?[0-9]+| ?[^\s\w]+|\s+(?!\S)|\s+"""
    )

def pre_tokenize(text):
    return [match.group() for match in GPT2_PATTERN.finditer(text)]

def apply_merge(byte_seq, pair, new_id):
    merged = []
    i = 0
    while i < len(byte_seq) - 1:
        if byte_seq[i] == pair[0] and byte_seq[i + 1] == pair[1]:
            merged.append(new_id)
            i += 2
        else:
            merged.append(byte_seq[i])
            i += 1

    if i == len(byte_seq) - 1:
        merged.append(byte_seq[i])

    return merged

    
class SpecialTokenHandler:
    def __init__(self):
        self.special_tokens = {}
        self.pattern = None

    def add_token(self, token_str, token_id):
        self.special_tokens[token_str] = token_id
        escaped = [re.escape(t) for t in sorted(self.special_tokens.keys(), key=len, reverse=True)]
        self.pattern = re.compile("|".join(escaped))

    def split_with_specials(self, text):
        if not self.pattern:
            return [(text, False)]

        parts = []
        last_end = 0
        for match in self.pattern.finditer(text):
            if match.start() > last_end:
                parts.append((text[last_end:match.start()], False))
            parts.append((match.group(), True))
            last_end = match.end()

        if last_end < len(text):
            parts.append((text[last_end:], False))

        return parts

class ProductionTokenizer:
    def __init__(self):
        self.merges = {}
        self.vocab = {i : bytes([i]) for i in range(256)}
        self.special_handler = SpecialTokenHandler()
        self.next_id = 256

    def normalize(self, text):
        return unicodedata.normalize("NFKC", text)

    def train(self, text, num_merges):
        text = self.normalize(text)
        chunks = pre_tokenize(text)
        chunk_bytes = [
            list(chunk.encode("utf-8")) for chunk in chunks
        ]

        for i in range(num_merges):
            pairs = Counter()
            for seq in chunk_bytes:
                for j in range(len(seq) - 1):
                    pairs[(seq[j], seq[j + 1])] += 1

            if not pairs:
                break

            best_pair = max(pairs, key=pairs.get)
            new_id = self.next_id
            self.next_id += 1
            self.merges[best_pair] = new_id
            self.vocab[new_id] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]

            chunk_bytes = [
                apply_merge(seq, best_pair, new_id) for seq in chunk_bytes
            ]

            merged_display = self.vocab[new_id]

            print(f"Merge {i + 1}: ({best_pair[0]} + {best_pair[1]} = {merged_display})")

    def add_special_tokens(self, token_str):
        token_id = self.next_id
        self.next_id += 1
        self.special_handler.add_token(token_str, token_id)
        self.vocab[token_id] = token_str.encode("utf-8")

    def encoder(self, text):
        text = self.normalize(text)
        parts = self.special_handler.split_with_specials(text)
        all_ids = []
        for part_text, is_speical in parts:
            if is_speical:
                all_ids.append(self.special_handler.special_tokens[part_text])
            else:
                for chunk in pre_tokenize(part_text):
                    byte_seq = list(chunk.encode("utf-8"))
                    for pair, new_id in self.merges.items():
                        byte_seq = apply_merge(byte_seq, pair, new_id)
                    all_ids.extend(byte_seq)

        return all_ids

    def decode(self, ids):
        byte_parts = []
        for token_id in ids:
            if token_id in self.vocab:
                byte_parts.append(self.vocab[token_id])

        return b"".join(byte_parts).decode("utf-8", errors="replace")  

    def vocab_size(self):
        return len(self.vocab)
    
    def get_token_bytes(self, token_id):
        return self.vocab.get(token_id, b"<?>")    
    
        

In [2]:
CORPUS = """
Natural language processing begins with tokenization, the process that converts raw text into discrete units a model can consume. Different designs trade vocabulary size against sequence length, and those choices permanently shape what the network sees during training and inference. Byte pair encoding starts from individual bytes or characters and repeatedly merges the most frequent adjacent pairs until a target vocabulary size is reached. The merge table becomes part of the tokenizer: encoding a new word replays those merges in

the same order that training discovered them. WordPiece also builds subword units, but it chooses merges by a likelihood-inspired score rather than raw co-occurrence counts alone. It marks continuation pieces with a special prefix so the model can distinguish word starts from mid-word fragments such as unhappiness becoming un and happi and ness. SentencePiece treats whitespace as an ordinary symbol and operates directly on Unicode, which makes the same pipeline usable for English, Chinese, and many other scripts without language-specific

pretokenization rules. Unigram language models reverse the BPE story by beginning with a large candidate vocabulary and pruning pieces that contribute least to the data likelihood. Large language models do not read letters or words in the human sense; they read integer token identifiers. A poorly designed tokenizer can explode the number of tokens needed for common phrases, shrink the effective context window, and raise serving cost for every prompt and completion. In practice engineers evaluate tokenizers on fertility, which

measures average tokens per word, on coverage of rare morphology, and on robustness to code, URLs, and multilingual mixtures. Balancing a larger embedding table against shorter sequences is a systems problem as much as a linguistic one. Consider a short story about a researcher who trains a small language model on diaries, manuals, and news articles. She watches the tokenizer invent compact pieces for frequent words like the and and, while rarer technical terms break into reusable stems and suffixes

that still preserve meaning across domains. Machine learning systems rely on careful data preparation, reproducible experiments, and clear evaluation metrics. Gradient descent adjusts parameters to reduce loss, while regularization methods such as dropout and weight decay help models generalize beyond the training set. Transformers attend over sequences with query, key, and value projections, enabling long-range dependencies without recurrence. Positional information can be added with sinusoidal encodings, learned embeddings, or relative schemes that better handle variable lengths. Software engineering for model

training involves datasets, dataloaders, checkpointing, mixed precision, and distributed strategies across multiple accelerators. Logging learning curves and inspecting failure cases often reveals tokenizer bugs long before architectural changes matter. Education materials explain that probability, linear algebra, and calculus form the mathematical backbone of modern deep learning. Students practice by implementing attention, residual connections, layer normalization, and feed-forward blocks from first principles. Open source communities share tokenizers, pretrained weights, and evaluation harnesses so practitioners can reproduce baselines and compare methods fairly.

Documentation that includes encoding examples and edge cases saves countless hours of debugging mysterious unknown tokens. Creative writing still matters for tokenizer corpora because fiction introduces dialogue, punctuation patterns, and narrative connectors that technical manuals underrepresent. Mixing genres produces merge rules that behave more stably when users switch between chat, code, and formal essays in the same session. Imagine cities connected by railways of information where each station is a token and each journey is a sentence. Compression algorithms decide

which stations to merge into express stops, leaving local stations for rare destinations that appear only occasionally in travel logs. Numerical text such as dates, measurements, and identifiers can fragment unpredictably if the training corpus never showed similar patterns. Including tables, formulas, and structured records encourages the tokenizer to keep useful digit groupings intact whenever possible. Multilingual corpora must respect script diversity: characters, syllables, and whitespace conventions vary widely. Underrepresenting a language forces that language into longer token sequences, which

quietly taxes latency and context for those users. During inference the model samples or searches over next-token distributions conditioned on previous identifiers. If tokenization is inconsistent between training and serving, the distribution the model learned no longer matches the inputs it receives, and quality collapses. Byte-level fallbacks guarantee that every Unicode string can be encoded without unknown symbols, at the cost of longer sequences for unfamiliar scripts. Hybrid designs combine a strong subword inventory with byte recovery for the long

tail of symbols and emojis. A practical exercise is to train a tiny BPE model on a few thousand words, inspect the earliest merges, and encode held-out sentences to see which fragments survive. Students usually discover that frequent function words become single tokens quickly, while long rare nouns remain compositional. Research papers discuss scaling laws, data quality, and alignment techniques that steer generative models toward helpful behavior. Yet even the strongest model remains constrained by the discrete vocabulary that sits

between human language and neural computation. The quick brown fox jumps over the lazy dog near the riverbank at dawn. Natural language processing begins with tokenization, the process that converts raw text into discrete units a model can consume. Scientists measure temperature, pressure, and humidity while calibrating sensitive laboratory instruments carefully. Programmers write tests, refactor modules, and review pull requests before merging changes into the main branch. Historians archive letters, maps, and photographs to reconstruct events that shaped communities over

centuries. SentencePiece treats whitespace as an ordinary symbol and operates directly on Unicode, which makes the same pipeline usable for English, Chinese, and many other scripts without language-specific pretokenization rules. Musicians rehearse melodies, harmonies, and rhythms until the ensemble performs with confidence and clarity. Farmers plant seeds, irrigate fields, and harvest crops according to seasonal weather patterns and soil conditions. Pilots check instruments, communicate with towers, and navigate routes across continents under changing skies. Consider a short story about a

researcher who trains a small language model on diaries, manuals, and news articles. Chefs prepare ingredients, balance flavors, and plate dishes that surprise guests with color and texture. Athletes train endurance, strength, and coordination through disciplined routines and recovery practices. Designers sketch interfaces, prototype interactions, and iterate layouts based on user feedback and analytics. Software engineering for model training involves datasets, dataloaders, checkpointing, mixed precision, and distributed strategies across multiple accelerators. The quick brown fox jumps over the lazy dog

near the riverbank at dawn. Scientists measure temperature, pressure, and humidity while calibrating sensitive laboratory instruments carefully. Programmers write tests, refactor modules, and review pull requests before merging changes into the main branch. Creative writing still matters for tokenizer corpora because fiction introduces dialogue, punctuation patterns, and narrative connectors that technical manuals underrepresent. Historians archive letters, maps, and photographs to reconstruct events that shaped communities over centuries. Musicians rehearse melodies, harmonies, and rhythms until the ensemble performs with confidence and

clarity. Farmers plant seeds, irrigate fields, and harvest crops according to seasonal weather patterns and soil conditions. Multilingual corpora must respect script diversity: characters, syllables, and whitespace conventions vary widely. Pilots check instruments, communicate with towers, and navigate routes across continents under changing skies. Chefs prepare ingredients, balance flavors, and plate dishes that surprise guests with color and texture. Athletes train endurance, strength, and coordination through disciplined routines and recovery practices. A practical exercise is to train a tiny BPE

model on a few thousand words, inspect the earliest merges, and encode held-out sentences to see which fragments survive. Designers sketch interfaces, prototype interactions, and iterate layouts based on user feedback and analytics. The quick brown fox jumps over the lazy dog near the riverbank at dawn. Scientists measure temperature, pressure, and humidity while calibrating sensitive laboratory instruments carefully. Byte pair encoding starts from individual bytes or characters and repeatedly merges the most frequent adjacent pairs until a target vocabulary

size is reached. Programmers write tests, refactor modules, and review pull requests before merging changes into the main branch. Historians archive letters, maps, and photographs to reconstruct events that shaped communities over centuries. Musicians rehearse melodies, harmonies, and rhythms until the ensemble performs with confidence and clarity. Large language models do not read letters or words in the human sense; they read integer token identifiers. Farmers plant seeds, irrigate fields, and harvest crops according to seasonal weather patterns and soil

conditions. Pilots check instruments, communicate with towers, and navigate routes across continents under changing skies. Chefs prepare ingredients, balance flavors, and plate dishes that surprise guests with color and texture. Machine learning systems rely on careful data preparation, reproducible experiments, and clear evaluation metrics. Athletes train endurance, strength, and coordination through disciplined routines and recovery practices. Designers sketch interfaces, prototype interactions, and iterate layouts based on user feedback and analytics. The quick brown fox jumps over the lazy dog near

the riverbank at dawn. Education materials explain that probability, linear algebra, and calculus form the mathematical backbone of modern deep learning. Scientists measure temperature, pressure, and humidity while calibrating sensitive laboratory instruments carefully. Programmers write tests, refactor modules, and review pull requests before merging changes into the main branch. Historians archive letters, maps, and photographs to reconstruct events that shaped communities over centuries. Imagine cities connected by railways of information where each station is a token and each journey is

a sentence. Musicians rehearse melodies, harmonies, and rhythms until the ensemble performs with confidence and clarity. Farmers plant seeds, irrigate fields, and harvest crops according to seasonal weather patterns and soil conditions. Pilots check instruments, communicate with towers, and navigate routes across continents under changing skies. During inference the model samples or searches over next-token distributions conditioned on previous identifiers. Chefs prepare ingredients, balance flavors, and plate dishes that surprise guests with color and texture. Athletes train endurance, strength, and

coordination through disciplined routines and recovery practices. Designers sketch interfaces, prototype interactions, and iterate layouts based on user feedback and analytics. Research papers discuss scaling laws, data quality, and alignment techniques that steer generative models toward helpful behavior. The quick brown fox jumps over the lazy dog near the riverbank at dawn. Scientists measure temperature, pressure, and humidity while calibrating sensitive laboratory instruments carefully. Programmers write tests, refactor modules, and review pull requests before merging changes into the main branch.

WordPiece also builds subword units, but it chooses merges by a likelihood-inspired score rather than raw co-occurrence counts alone. Historians archive letters, maps, and photographs to reconstruct events that shaped communities over centuries. Musicians rehearse melodies, harmonies, and rhythms until the ensemble performs with confidence and clarity. Farmers plant seeds, irrigate fields, and harvest crops according to seasonal weather patterns and soil conditions. In practice engineers evaluate tokenizers on fertility, which measures average tokens per word, on coverage of rare

morphology, and on robustness to code, URLs, and multilingual mixtures. Pilots check instruments, communicate with towers, and navigate routes across continents under changing skies. Chefs prepare ingredients, balance flavors, and plate dishes that surprise guests with color and texture. Athletes train endurance, strength, and coordination through disciplined routines and recovery practices. Transformers attend over sequences with query, key, and value projections, enabling long-range dependencies without recurrence. Designers sketch interfaces, prototype interactions, and iterate layouts based on user feedback and analytics.

The quick brown fox jumps over the lazy dog near the riverbank at dawn. Scientists measure temperature, pressure, and humidity while calibrating sensitive laboratory instruments carefully. Open source communities share tokenizers, pretrained weights, and evaluation harnesses so practitioners can reproduce baselines and compare methods fairly. Programmers write tests, refactor modules, and review pull requests before merging changes into the main branch. Historians archive letters, maps, and photographs to reconstruct events that shaped communities over centuries. Musicians rehearse melodies, harmonies, and

rhythms until the ensemble performs with confidence and clarity. Numerical text such as dates, measurements, and identifiers can fragment unpredictably if the training corpus never showed similar patterns. Farmers plant seeds, irrigate fields, and harvest crops according to seasonal weather patterns and soil conditions. Pilots check instruments, communicate with towers, and navigate routes across continents under changing skies. Chefs prepare ingredients, balance flavors, and plate dishes that surprise guests with color and texture. Byte-level fallbacks guarantee that every Unicode string

can be encoded without unknown symbols, at the cost of longer sequences for unfamiliar scripts. Athletes train endurance, strength, and coordination through disciplined routines and recovery practices. Designers sketch interfaces, prototype interactions, and iterate layouts based on user feedback and analytics. The quick brown fox jumps over the lazy dog near the riverbank at dawn. Natural language processing begins with tokenization, the process that converts raw text into discrete units a model can consume. Scientists measure temperature, pressure, and humidity

while calibrating sensitive laboratory instruments carefully. Programmers write tests, refactor modules, and review pull requests before merging changes into the main branch. Historians archive letters, maps, and photographs to reconstruct events that shaped communities over centuries. SentencePiece treats whitespace as an ordinary symbol and operates directly on Unicode, which makes the same pipeline usable for English, Chinese, and many other scripts without language-specific pretokenization rules. Musicians rehearse melodies, harmonies, and rhythms until the ensemble performs with confidence and clarity. Farmers

plant seeds, irrigate fields, and harvest crops according to seasonal weather patterns and soil conditions. Pilots check instruments, communicate with towers, and navigate routes across continents under changing skies. Consider a short story about a researcher who trains a small language model on diaries, manuals, and news articles. Chefs prepare ingredients, balance flavors, and plate dishes that surprise guests with color and texture. Athletes train endurance, strength, and coordination through disciplined routines and recovery practices. Designers sketch interfaces, prototype interactions,

and iterate layouts based on user feedback and analytics. Software engineering for model training involves datasets, dataloaders, checkpointing, mixed precision, and distributed strategies across multiple accelerators. The quick brown fox jumps over the lazy dog near the riverbank at dawn. Scientists measure temperature, pressure, and humidity while calibrating sensitive laboratory instruments carefully. Programmers write tests, refactor modules, and review pull requests before merging changes into the main branch. Creative writing still matters for tokenizer corpora because fiction introduces dialogue, punctuation

patterns, and narrative connectors that technical manuals underrepresent. Historians archive letters, maps, and photographs to reconstruct events that shaped communities over centuries. Musicians rehearse melodies, harmonies, and rhythms until the ensemble performs with confidence and clarity. Farmers plant seeds, irrigate fields, and harvest crops according to seasonal weather patterns and soil conditions. Multilingual corpora must respect script diversity: characters, syllables, and whitespace conventions vary widely. Pilots check instruments, communicate with towers, and navigate routes across continents under changing skies. Chefs

prepare ingredients, balance flavors, and plate dishes that surprise guests with color and texture. Athletes train endurance, strength, and coordination through disciplined routines and recovery practices. A practical exercise is to train a tiny BPE model on a few thousand words, inspect the earliest merges, and encode held-out sentences to see which fragments survive. Designers sketch interfaces, prototype interactions, and iterate layouts based on user feedback and analytics. The quick brown fox jumps over the lazy dog near the riverbank

at dawn. Scientists measure temperature, pressure, and humidity while calibrating sensitive laboratory instruments carefully. Byte pair encoding starts from individual bytes or characters and repeatedly merges the most frequent adjacent pairs until a target vocabulary size is reached. Programmers write tests, refactor modules, and review pull requests before merging changes into the main branch. Historians archive letters, maps, and photographs to reconstruct events that shaped communities over centuries. Musicians rehearse melodies, harmonies, and rhythms until the ensemble performs with confidence

and clarity. Large language models do not read letters or words in the human sense; they read integer token identifiers. Farmers plant seeds, irrigate fields, and harvest crops according to seasonal weather patterns and soil conditions. Pilots check instruments, communicate with towers, and navigate routes across continents under changing skies. Chefs prepare ingredients, balance flavors, and plate dishes that surprise guests with color and texture. Machine learning systems rely on careful data preparation, reproducible experiments, and clear evaluation metrics. Athletes

train endurance, strength, and coordination through disciplined routines and recovery practices. Designers sketch interfaces, prototype interactions, and iterate layouts based on user feedback and analytics. The quick brown fox jumps over the lazy dog near the riverbank at dawn. Education materials explain that probability, linear algebra, and calculus form the mathematical backbone of modern deep learning. Scientists measure temperature, pressure, and humidity while calibrating sensitive laboratory instruments carefully. Programmers write tests, refactor modules, and review pull requests before merging changes

into the main branch. Historians archive letters, maps, and photographs to reconstruct events that shaped communities over centuries. Imagine cities connected by railways of information where each station is a token and each journey is a sentence. Musicians rehearse melodies, harmonies, and rhythms until the ensemble performs with confidence and clarity. Farmers plant seeds, irrigate fields, and harvest crops according to seasonal weather patterns and soil conditions. Pilots check instruments, communicate with towers, and navigate routes across continents under changing

skies. During inference the model samples or searches over next-token distributions conditioned on previous identifiers. Chefs prepare ingredients, balance flavors, and plate dishes that surprise guests with color and texture. Athletes train endurance, strength, and coordination through disciplined routines and recovery practices. Designers sketch interfaces, prototype interactions, and iterate layouts based on user feedback and analytics. Research papers discuss scaling laws, data quality, and alignment techniques that steer generative models toward helpful behavior. The quick brown fox jumps over the

lazy dog near the riverbank at dawn. Scientists measure temperature, pressure, and humidity while calibrating sensitive laboratory instruments carefully. Programmers write tests, refactor modules, and review pull requests before merging changes into the main branch. WordPiece also builds subword units, but it chooses merges by a likelihood-inspired score rather than raw co-occurrence counts alone. Historians archive letters, maps, and photographs to reconstruct events that shaped communities over centuries. Musicians rehearse melodies, harmonies, and rhythms until the ensemble performs with confidence

and clarity. Farmers plant seeds, irrigate fields, and harvest crops according to seasonal weather patterns and soil conditions. In practice engineers evaluate tokenizers on fertility, which measures average tokens per word, on coverage of rare morphology, and on robustness to
"""
TEXT = "Hello world! This is a test."
print("CORPUS words:", len(CORPUS.split()))


CORPUS words: 3000


In [3]:
tok = ProductionTokenizer()
tok.train(CORPUS, num_merges=100)

Merge 1: (97 + 110 = b'an')
Merge 2: (101 + 114 = b'er')
Merge 3: (101 + 115 = b'es')
Merge 4: (105 + 110 = b'in')
Merge 5: (32 + 116 = b' t')
Merge 6: (101 + 110 = b'en')
Merge 7: (97 + 116 = b'at')
Merge 8: (114 + 101 = b're')
Merge 9: (32 + 99 = b' c')
Merge 10: (111 + 110 = b'on')
Merge 11: (32 + 115 = b' s')
Merge 12: (32 + 256 = b' an')
Merge 13: (267 + 100 = b' and')
Merge 14: (111 + 114 = b'or')
Merge 15: (32 + 112 = b' p')
Merge 16: (105 + 116 = b'it')
Merge 17: (97 + 114 = b'ar')
Merge 18: (32 + 109 = b' m')
Merge 19: (260 + 104 = b' th')
Merge 20: (116 + 115 = b'ts')
Merge 21: (97 + 108 = b'al')
Merge 22: (32 + 119 = b' w')
Merge 23: (259 + 103 = b'ing')
Merge 24: (97 + 99 = b'ac')
Merge 25: (114 + 111 = b'ro')
Merge 26: (32 + 100 = b' d')
Merge 27: (116 + 105 = b'ti')
Merge 28: (32 + 108 = b' l')
Merge 29: (32 + 98 = b' b')
Merge 30: (32 + 263 = b' re')
Merge 31: (32 + 102 = b' f')
Merge 32: (257 + 115 = b'ers')
Merge 33: (32 + 259 = b' in')
Merge 34: (274 + 101 = b' the')


In [ ]:
texts = [
    ("English", "hello"),
    ("Chinese", "你好"),
    ("Japanese", "こんにちは"),
    ("Emoji", "🔥🌍"),
    ("Mixed", "hello你好🔥"),
    ("Code", "def f(x):"),
]

for label, text in texts:
    b = list(text.encode("utf-8"))
    print(f"{label:10s}: {len(text):2d} chars -> {len(b):2d} bytes -> {b[:16]}{'...' if len(b) > 16 else ''}")
